# Definindo sistema


In [1]:
import numpy as np
import torch

from lib.physics import simulate
from lib.plots import plot_tanks

t = np.linspace(0, 300, 300)
t_tensor = torch.tensor(t, dtype=torch.float32, requires_grad=True).unsqueeze(1)
y0 = np.array([0.5, 0.5])
h1_sim, h2_sim = simulate(y0, t_eval=t)

plot_tanks(t, [h1_sim, h2_sim], ["$h_1$ (sim)", "$h_2$ (sim)"], filename="sim-tanks")

# Add noise
h1_exp = h1_sim + np.random.normal(0, 1, len(t)) * 0.1
h2_exp = h2_sim + np.random.normal(0, 1, len(t)) * 0.1

plot_tanks(
    t, [h1_exp, h2_exp], ["$h_1$ (exp)", "$h_2$ (exp)"], scatter=2, filename="exp-tanks"
)

# from lib.physics import F
# from matplotlib import pyplot as plt

# plt.figure(figsize=(10, 4), layout="constrained")
# plt.plot(t, F(t), label="F")
# plt.show()


# Definindo rede neural


In [2]:
from lib.BaseModel import BaseModel


def getNewModel():
    return BaseModel(
        max_input=float(torch.max(t_tensor)),
        max_output=float(torch.max(h1_exp)),
    )


In [3]:
from lib.physics import edo_torch
from lib.utils import dydx, mean_square

h1_exp = torch.tensor(h1_exp, dtype=torch.float32)
h2_exp = torch.tensor(h2_exp, dtype=torch.float32)


def loss_fn(model, t: torch.Tensor):
    # Loss das EDOs
    Y_pred = model(t)
    h1_pred, h2_pred = Y_pred[:, 0], Y_pred[:, 1]

    dh1dt_pinn, dh2dt_pinn = dydx(t, h1_pred), dydx(t, h2_pred)
    dh1dt_edo, dh2dt_edo = edo_torch(t, [h1_pred, h2_pred])

    loss_EDO1 = mean_square(dh1dt_pinn - dh1dt_edo)
    loss_EDO2 = mean_square(dh2dt_pinn - dh2dt_edo)

    # Loss das condições iniciais
    t0 = torch.tensor([[0.0]], requires_grad=True)
    Y0 = model(t0)
    h1_0, h2_0 = Y0[:, 0], Y0[:, 1]

    loss_ic1 = mean_square(h1_0 - y0[0])
    loss_ic2 = mean_square(h2_0 - y0[1])

    # Loss dos dados
    loss_data_h1 = mean_square(h1_pred - h1_exp)
    loss_data_h2 = mean_square(h2_pred - h2_exp)

    # Loss total
    loss_total = (
        loss_EDO1 + loss_EDO2 + loss_data_h1 + loss_data_h2 + loss_ic1 + loss_ic2
    )

    return loss_total


# Otimizando hiperparâmetros dos métodos


In [4]:
n_execuções = 50
target_loss = 0.1
trials = 100

Adam_study_path = "../results/Adam-studies.hkl"
Adam_speeds_path = "../results/Adam-speeds.hkl"
Adam_performance_path = "../results/Adam-performance.hkl"
Adam_model_path = "../results/Adam-model.pt"
Adam_full_model_path = "../results/Adam-full-model.pt"


GA_study_path = "../results/GA-studies.hkl"
GA_speeds_path = "../results/GA-speeds.hkl"
GA_performance_path = "../results/GA-performance.hkl"
GA_model_path = "../results/GA-model.pt"
GA_full_model_path = "../results/GA-full-model.pt"

GA_and_Adam_speeds_path = "../results/GA-and-Adam-speeds.hkl"
GA_and_Adam_performance_path = "../results/GA-and-Adam-performance.hkl"
GA_and_Adam_model_path = "../results/GA-and-Adam-model.pt"
GA_and_Adam_full_model_path = "../results/GA-and-Adam-full-model.pt"


def count_fails(losses):
    return np.sum(np.array(losses) > target_loss)


## Adam


In [5]:
import optuna

from lib.optuna import study
from lib.utils import train


def objective(trial: optuna.Trial):
    torch.manual_seed(42)
    test_model = getNewModel()

    lr = trial.suggest_float("lr", 1e-15, 1)
    beta1 = trial.suggest_float("beta1", 1e-10, 1)
    beta2 = trial.suggest_float("beta2", 1e-10, 1)

    optimizer = torch.optim.Adam(test_model.parameters(), lr=lr, betas=(beta1, beta2))

    return train(test_model, loss_fn, optimizer, 1000, t_tensor)


best_Adam_params = study(objective, trials, "Adam-study", Adam_study_path)


[I 2025-02-11 14:54:34,205] A new study created in memory with name: Adam-study


Otimização realizada anteriormente. Recuperando valores...
Imprimindo resultado:
  Valor do Loss: 0.02227054163813591
  hiperparâmetros:
    lr: 0.055831326000097936
    beta1: 0.9061889946678847
    beta2: 0.9983657218540035


## GA


In [6]:
import pygad
from pygad.torchga import torchga

GA_model = getNewModel()


def fitness_func(ga_instance, solution, solution_idx):
    model_weights_dict = torchga.model_weights_as_dict(
        model=GA_model, weights_vector=solution
    )
    GA_model.load_state_dict(model_weights_dict)

    GA_model.eval()
    loss = loss_fn(GA_model, t_tensor)

    # Quanto menor o loss, maior o fitness
    return -loss.item()


In [7]:
def objective(trial: optuna.Trial):
    torch.manual_seed(42)
    test_model = getNewModel()

    parent_selection_type = trial.suggest_categorical(
        "parent_selection_type", ["sss", "rws", "sus", "rank", "random", "tournament"]
    )
    keep_elitism = trial.suggest_int("keep_elitism", 0, 10)
    num_parents_mating = trial.suggest_int("num_parents_mating", 2, 10)

    crossover_type = trial.suggest_categorical(
        "crossover_type", ["single_point", "two_points", "uniform", "scattered", None]
    )
    crossover_probability = trial.suggest_float("crossover_probability", 0, 1)

    mutation_type = trial.suggest_categorical(
        "mutation_type", ["random", "swap", "inversion", "scramble", "adaptive", None]
    )
    mutation_probability = trial.suggest_float("mutation_probability", 0, 1)

    # Configura o TorchGA para criar populações baseadas no modelo
    torch_ga = torchga.TorchGA(model=test_model, num_solutions=50)

    # Configura o algoritmo genético
    ga_instance = pygad.GA(
        # Configurações
        initial_population=torch_ga.population_weights,  # População inicial
        fitness_func=fitness_func,  # Função de aptidão
        num_generations=80,  # Número de gerações
        random_seed=42,
        init_range_low=-4,
        init_range_high=4,
        # Parâmetros para otimizar
        parent_selection_type=parent_selection_type,
        keep_elitism=keep_elitism,
        num_parents_mating=num_parents_mating,
        crossover_type=crossover_type,  # type: ignore
        crossover_probability=crossover_probability,
        mutation_type=mutation_type,  # type: ignore
        mutation_probability=(
            mutation_probability if mutation_type != "adaptive" else [0.8, 0.1]
        ),
    )

    # Executa o algoritmo genético
    ga_instance.run()

    _, best_solution_fitness, _ = ga_instance.best_solution()

    return float(best_solution_fitness)


best_GA_params = study(
    objective, trials, "GA-study", GA_study_path, study_direction="maximize"
)


[I 2025-02-11 14:54:34,228] A new study created in memory with name: GA-study


Otimização realizada anteriormente. Recuperando valores...
Imprimindo resultado:
  Valor do Loss: -0.11438300460577011
  hiperparâmetros:
    parent_selection_type: sss
    keep_elitism: 4
    num_parents_mating: 6
    crossover_type: single_point
    crossover_probability: 0.38405026194985326
    mutation_type: random
    mutation_probability: 0.05416772930795577


# Testes de velocidade


## Adam


In [8]:
from lib.test_speed import test_train_speed


def Adam_train(model, seed):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_Adam_params["lr"],
        betas=(best_Adam_params["beta1"], best_Adam_params["beta2"]),
    )
    loss_value = train(model, loss_fn, optimizer, 5000, t_tensor, target_loss)
    return loss_value


Adam_times, Adam_losses, Adam_model = test_train_speed(
    Adam_train, getNewModel, n_execuções, Adam_speeds_path, Adam_model_path
)


print(f"Média do tempo (Adam): {np.mean(Adam_times):.3f}s")
print("Tentativas falhadas:", count_fails(Adam_losses))


Recuperando testes anteriores
Média do tempo (Adam): 0.184s
Tentativas falhadas: 0


In [9]:
@torch.inference_mode()
def test_model(model):
    y = model(t_tensor)
    return [y[:, 0], y[:, 1]]


h1_pinn_adam, h2_pinn_adam = test_model(Adam_model)

# Gráfico
plot_tanks(
    t,
    (h1_sim, h2_sim, h1_pinn_adam, h2_pinn_adam),
    ["$h_1$ (sim)", "$h_2$ (sim)", "$h_1$ (PINN - Adam)", "$h_2$ (PINN - Adam)"],
    filename="velocity-test-adam",
)


## Algoritmo Genético


In [10]:
def on_generation(ga_instance):
    if ga_instance.best_solution()[1] > -target_loss:
        print("Chegou no loss alvo antes de terminar as gerações!")
        return "stop"

    return None


if best_GA_params["mutation_type"] == "adaptive":
    best_GA_params["mutation_probability"] = [0.8, 0.1]


def GA_train(model, seed):
    torch_ga = torchga.TorchGA(model=model, num_solutions=200)
    ga_instance = pygad.GA(
        initial_population=torch_ga.population_weights,
        fitness_func=fitness_func,
        random_seed=seed,
        num_generations=500,
        init_range_low=-4,
        init_range_high=4,
        on_generation=on_generation,
        **best_GA_params,
    )
    ga_instance.run()
    best_solution, best_solution_fitness, _ = ga_instance.best_solution()

    model_weights_dict = torchga.model_weights_as_dict(
        model=model, weights_vector=best_solution
    )
    model.load_state_dict(model_weights_dict)

    return -best_solution_fitness


GA_times, GA_losses, GA_model = test_train_speed(
    GA_train, getNewModel, n_execuções, GA_speeds_path, GA_model_path
)

print(f"Média do tempo (GA): {np.mean(GA_times):.3f}s")
print("Tentativas falhadas:", count_fails(GA_losses))


Recuperando testes anteriores
Média do tempo (GA): 128.464s
Tentativas falhadas: 11


In [11]:
h1_pinn_ga, h2_pinn_ga = test_model(GA_model)

# Gráfico
plot_tanks(
    t,
    (h1_sim, h2_sim, h1_pinn_ga, h2_pinn_ga),
    ["$h_1$ (sim)", "$h_2$ (sim)", "$h_1$ (PINN - AG)", "$h_2$ (PINN - AG)"],
    filename="velocity-test-ga",
)

# for nome, param in GA_model.named_parameters():
#    print(f"Nome: {nome}")
#    print(f"Valor: {param}")


## GA + Adam


In [12]:
def GA_and_Adam_train(model, seed):
    torch_ga = torchga.TorchGA(model=model, num_solutions=25)
    ga_instance = pygad.GA(
        initial_population=torch_ga.population_weights,
        fitness_func=fitness_func,
        random_seed=seed,
        num_generations=3,
        init_range_low=-4,
        init_range_high=4,
        **best_GA_params,
    )
    ga_instance.run()
    best_solution, _, _ = ga_instance.best_solution()

    model_weights_dict = torchga.model_weights_as_dict(
        model=model, weights_vector=best_solution
    )
    model.load_state_dict(model_weights_dict)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_Adam_params["lr"],
        betas=(best_Adam_params["beta1"], best_Adam_params["beta2"]),
    )
    loss_value = train(model, loss_fn, optimizer, 5000, t_tensor, target_loss)
    return loss_value


GA_and_Adam_times, GA_and_Adam_losses, GA_and_Adam_model = test_train_speed(
    GA_and_Adam_train,
    getNewModel,
    n_execuções,
    GA_and_Adam_speeds_path,
    GA_and_Adam_model_path,
)

print(f"Média do tempo (GA + Adam): {np.mean(GA_and_Adam_times):.3f}s")
print("Tentativas falhadas:", count_fails(GA_and_Adam_losses))

h1_pinn_ga_adam, h2_pinn_ga_adam = test_model(GA_and_Adam_model)

# Gráfico
plot_tanks(
    t,
    (h1_sim, h2_sim, h1_pinn_ga_adam, h2_pinn_ga_adam),
    [
        "$h_1$ (sim)",
        "$h_2$ (sim)",
        "$h_1$ (PINN - AG + Adam)",
        "$h_2$ (PINN - AG + Adam)",
    ],
    filename="velocity-test-ga-adam",
)


Recuperando testes anteriores
Média do tempo (GA + Adam): 0.412s
Tentativas falhadas: 0


## Salvando resultados


In [13]:
import seaborn as sns
from drawarrow import fig_arrow
from matplotlib import pyplot as plt

from lib.plots import colors, plot_density, save_or_show

Adam_losses = np.array(Adam_losses)
Adam_times = np.array(Adam_times)
GA_losses = np.array(GA_losses)
GA_times = np.array(GA_times)

# Remove os valores que falharam
successful_Adam = Adam_times[Adam_losses < target_loss]
successful_GA = GA_times[GA_losses < target_loss]

print("Tentativas com sucesso (Adam):", len(successful_Adam))
print("Tentativas com sucesso (GA):", len(successful_GA))

values = [successful_Adam, successful_GA]
labels = ["Adam", "AG"]


def zoom(plt):
    fig_arrow(
        head_position=(0.24, 0.8),
        tail_position=(0.13, 0.7),
        width=3,
        radius=0.15,
        color="darkred",
        mutation_scale=1.2,
    )
    sub_axes = plt.axes((0.25, 0.7, 0.25, 0.20))
    # sub_axes.set_xticks([])
    # sub_axes.set_yticks([])
    sns.histplot(
        successful_Adam,
        kde=True,
        bins=15,
        color=colors[0],
        ax=sub_axes,
        stat="density",
    )
    sub_axes.axvline(
        np.mean(successful_Adam[-1]),
        linestyle="--",
        linewidth=1.5,
        color=colors[0],
    )
    sub_axes.set_ylabel(None)


plot_density(values, labels, filename="density", extra=zoom)

values = [Adam_times, GA_and_Adam_times]
labels = ["Adam", "AG + Adam"]


plot_density(values, labels, filename="density-2")


fig, axes = plt.subplots(3, 1, figsize=(10, 10))

# Dados para os diferentes métodos
methods = [
    (h1_pinn_adam, h2_pinn_adam, "PINN - Adam"),
    (h1_pinn_ga, h2_pinn_ga, "PINN - AG"),
    (h1_pinn_ga_adam, h2_pinn_ga_adam, "PINN - AG + Adam"),
]

# Iterar sobre os subplots e dados
for ax, (h1_pinn, h2_pinn, label) in zip(axes, methods):
    ax.plot(t, h1_sim, label="$h_1$ (sim)", color="tab:blue", linewidth=3, alpha=0.5)
    ax.plot(t, h2_sim, label="$h_2$ (sim)", color="tab:orange", linewidth=3, alpha=0.5)
    ax.plot(
        t,
        h1_pinn,
        label=f"$h_1$ ({label})",
        color="tab:blue",
        linestyle="--",
        linewidth=3,
    )
    ax.plot(
        t,
        h2_pinn,
        label=f"$h_2$ ({label})",
        color="tab:orange",
        linestyle="--",
        linewidth=3,
    )
    ax.legend()
    ax.grid(True)

fig.supxlabel("Tempo / s")
fig.supylabel("Nível / cm")
plt.tight_layout()
save_or_show("velocity-test-all")


Tentativas com sucesso (Adam): 50
Tentativas com sucesso (GA): 39


# Testes de desempenho


## Adam


In [14]:
def Adam_train(model, seed):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_Adam_params["lr"],
        betas=(best_Adam_params["beta1"], best_Adam_params["beta2"]),
    )
    loss_value = train(model, loss_fn, optimizer, 50000, t_tensor, early_stopping=500)
    return loss_value


full_Adam_times, full_Adam_losses, full_Adam_model = test_train_speed(
    Adam_train, getNewModel, 1, Adam_performance_path, Adam_full_model_path
)


h1_pinn_adam, h2_pinn_adam = test_model(full_Adam_model)

# Gráfico
plot_tanks(
    t,
    (h1_sim, h2_sim, h1_pinn_adam, h2_pinn_adam),
    ["$h_1$ (sim)", "$h_2$ (sim)", "$h_1$ (PINN - Adam)", "$h_2$ (PINN - Adam)"],
    filename="performance-test-adam",
)


def get_validation_loss(h1_model, h2_model):
    return float(
        mean_square(h1_model - torch.tensor(h1_sim))
        + mean_square(h2_model - torch.tensor(h2_sim))
    )


full_Adam_validation_loss = get_validation_loss(h1_pinn_adam, h2_pinn_adam)
print(full_Adam_losses[0], full_Adam_validation_loss)


Recuperando testes anteriores
0.03788122907280922 0.01754283761380982


## GA


In [15]:
def GA_train(model, seed):
    torch_ga = torchga.TorchGA(model=model, num_solutions=1000)
    ga_instance = pygad.GA(
        initial_population=torch_ga.population_weights,
        fitness_func=fitness_func,
        random_seed=seed,
        num_generations=50000,
        init_range_low=-4,
        init_range_high=4,
        stop_criteria="saturate_50",
        **best_GA_params,
    )
    ga_instance.run()
    best_solution, best_solution_fitness, _ = ga_instance.best_solution()

    model_weights_dict = torchga.model_weights_as_dict(
        model=model, weights_vector=best_solution
    )
    model.load_state_dict(model_weights_dict)

    return -best_solution_fitness


full_GA_times, full_GA_losses, full_GA_model = test_train_speed(
    GA_train, getNewModel, 1, GA_performance_path, GA_full_model_path
)

h1_pinn_ga, h2_pinn_ga = test_model(full_GA_model)

# Gráfico
plot_tanks(
    t,
    (h1_sim, h2_sim, h1_pinn_ga, h2_pinn_ga),
    ["$h_1$ (sim)", "$h_2$ (sim)", "$h_1$ (PINN - AG)", "$h_2$ (PINN - AG)"],
    filename="performance-test-ga",
)

full_GA_validation_loss = get_validation_loss(h1_pinn_ga, h2_pinn_ga)
print(full_GA_losses[0], full_GA_validation_loss)


Recuperando testes anteriores
0.06605106592178345 0.04948143460895146


## GA + Adam


In [16]:
def GA_and_Adam_train(model, seed):
    torch_ga = torchga.TorchGA(model=model, num_solutions=1000)
    ga_instance = pygad.GA(
        initial_population=torch_ga.population_weights,
        fitness_func=fitness_func,
        random_seed=seed,
        num_generations=50,
        init_range_low=-4,
        init_range_high=4,
        **best_GA_params,
    )
    ga_instance.run()
    best_solution, _, _ = ga_instance.best_solution()

    model_weights_dict = torchga.model_weights_as_dict(
        model=model, weights_vector=best_solution
    )
    model.load_state_dict(model_weights_dict)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_Adam_params["lr"],
        betas=(best_Adam_params["beta1"], best_Adam_params["beta2"]),
    )
    loss_value = train(model, loss_fn, optimizer, 50000, t_tensor, early_stopping=500)
    return loss_value


full_GA_and_Adam_times, full_GA_and_Adam_losses, full_GA_and_Adam_model = (
    test_train_speed(
        GA_and_Adam_train,
        getNewModel,
        1,
        GA_and_Adam_performance_path,
        GA_and_Adam_full_model_path,
    )
)

h1_pinn_ga_adam, h2_pinn_ga_adam = test_model(full_GA_and_Adam_model)

# Gráfico
plot_tanks(
    t,
    (h1_sim, h2_sim, h1_pinn_ga_adam, h2_pinn_ga_adam),
    [
        "$h_1$ (sim)",
        "$h_2$ (sim)",
        "$h_1$ (PINN - AG + Adam)",
        "$h_2$ (PINN - AG + Adam)",
    ],
    filename="performance-test-ga-adam",
)

full_GA_and_Adam_validation_loss = get_validation_loss(h1_pinn_ga_adam, h2_pinn_ga_adam)
print(full_GA_and_Adam_losses[0], full_GA_and_Adam_validation_loss)


Recuperando testes anteriores
0.02281433530151844 0.001475765654314282


## Generate performance table and graph


In [17]:
from lib.utils import save_performance_table

methods = ["Adam", "AG", "AG + Adam"]
training_losses = [full_Adam_losses[0], full_GA_losses[0], full_GA_and_Adam_losses[0]]
validation_losses = [
    full_Adam_validation_loss,
    full_GA_validation_loss,
    full_GA_and_Adam_validation_loss,
]

save_performance_table(
    methods,
    training_losses,
    validation_losses,
    "../results/performance_table.tex",
    "tab:performance",
)

fig, axes = plt.subplots(3, 1, figsize=(10, 10))

# Dados para os diferentes métodos
methods = [
    (h1_pinn_adam, h2_pinn_adam, "PINN - Adam"),
    (h1_pinn_ga, h2_pinn_ga, "PINN - AG"),
    (h1_pinn_ga_adam, h2_pinn_ga_adam, "PINN - AG + Adam"),
]

# Iterar sobre os subplots e dados
for ax, (h1_pinn, h2_pinn, label) in zip(axes, methods):
    ax.plot(t, h1_sim, label="$h_1$ (sim)", color="tab:blue", linewidth=3, alpha=0.5)
    ax.plot(t, h2_sim, label="$h_2$ (sim)", color="tab:orange", linewidth=3, alpha=0.5)
    ax.plot(
        t,
        h1_pinn,
        label=f"$h_1$ ({label})",
        color="tab:blue",
        linestyle="--",
        linewidth=3,
    )
    ax.plot(
        t,
        h2_pinn,
        label=f"$h_2$ ({label})",
        color="tab:orange",
        linestyle="--",
        linewidth=3,
    )
    ax.legend()
    ax.grid(True)

fig.supxlabel("Tempo / s")
fig.supylabel("Nível / cm")
plt.tight_layout()
save_or_show("performance-test-all")
